# Calculo Financeiro - BACEN
Este notebook demonstra a construção de uma API em Python que consome dados públicos do Banco Central (BACEN), realiza cálculos financeiros (como juros compostos e variação acumulada) e disponibiliza os resultados via endpoints.

# Tranformando o JSON em DataFrame

In [ ]:
# Transforma cada histórico em um DataFrame e adiciona a coluna 'indicador'
df_ipca = pd.DataFrame(resultados['IPCA']['historico'])
df_ipca['indicador'] = 'IPCA'

df_selic = pd.DataFrame(resultados['SELIC']['historico'])
df_selic['indicador'] = 'SELIC'

In [ ]:
# Junta os dois indicadores em um único DataFrame
df_historico = pd.concat([df_ipca, df_selic], ignore_index=True)

# Converte a coluna 'data' para datetime
df_historico['data'] = pd.to_datetime(df_historico['data'])


In [ ]:
# Pega as estatísticas de cada indicador e cria DataFrame
df_estatisticas = pd.DataFrame([
    {"indicador": "IPCA", **resultados['IPCA']['estatisticas']},
    {"indicador": "SELIC", **resultados['SELIC']['estatisticas']}
])

# Limpeza e ajustes

In [ ]:
# Preencher valores ausentes
df_historico['valor'] = df_historico['valor'].ffill()  
df_historico = df_historico.drop_duplicates(subset=['data', 'indicador'])  # remove duplicatas

# Remove outliers usando Z-score
df_historico['z'] = df_historico.groupby('indicador')['valor'].transform(lambda x: zscore(x, nan_policy='omit'))
df_historico = df_historico[df_historico['z'].abs() < 3].drop(columns='z')

# Padronização de datas 

In [ ]:
# Separa os indices 
df_ipca = df_historico[df_historico['indicador'] == 'IPCA'].copy()
df_selic = df_historico[df_historico['indicador'] == 'SELIC'].copy()

# Garantir ordenação
df_selic['data'] = pd.to_datetime(df_selic['data'])
df_selic = df_selic.set_index('data')

df_ipca['data'] = pd.to_datetime(df_ipca['data'])
df_ipca = df_ipca.set_index('data')

## Transformando SELIC em dado mensalizado 

In [ ]:
# Agrupar por mês e pegar o último valor 
df_selic_mensal = (
    df_selic
    .resample('ME')   # Month End
    .last()
)

# Transformar data para dia 01
df_selic_mensal.index = df_selic_mensal.index.to_period('M').to_timestamp()

# MODELO SARIMA IPCA

In [ ]:
# Converte o índice para o tipo DatetimeIndex do pandas.
df_ipca.index = pd.to_datetime(df_ipca.index)
df_ipca = df_ipca.asfreq('MS')   # MS = Month Start


In [ ]:
# Testa estacionariedade
teste_adf(df_ipca['valor'])

# Ajusta modelo
# Após testes comparativos baseados no critério AIC e diagnóstico dos resíduos, o modelo SARIMA(1,0,1) mostrou-se o mais adequado para a série do IPCA.
modelo_ipca = ajustar_sarima(
    df_ipca['valor'],
    ordem=(1,0,1),
    ordem_sazonal=(0,0,0,12)
)

# Faze previsão
projecao_ipca = prever_ipca(modelo_ipca, passos=48)

# Plotar
plotar_previsao(df_ipca['valor'], projecao_ipca)

# MODELO ARIMA SELIC

In [ ]:
# Converte o índice para o tipo DatetimeIndex do pandas.
df_selic_mensal.index = pd.to_datetime(df_selic_mensal.index)
df_selic_mensal = df_selic_mensal.asfreq('MS')   # MS = Month Start

In [ ]:
# Testar estacionariedade
print("Teste ADF - Série em nível")
testar_estacionariedade(df_selic_mensal['valor'])

# Ajustar modelo ARIMA
modelo_arima = ajustar_arima_diferenciado(df_selic_mensal['valor'])

# Projeção
projecao_selic = projetar_arima(
    modelo=modelo_arima,
    serie=df_selic_mensal['valor'],
    ano_final=2030
)

# Plotar
plotar_projecao(df_selic_mensal, projecao_selic)

# Junção da SELIC e IPCA

## Junção dos DataFrames Passdados Mensalizados 

In [ ]:
# Concatenando os dataframes passados
df_macro_observados = pd.concat([df_ipca, df_selic_mensal])

# Garantir que o índice é datetime
df_macro_observados.index = pd.to_datetime(df_macro_observados.index)

# Ordenar o índice
df_macro_observados = df_macro_observados.sort_index()

## Junção dos DataFrames Futuros projetados Mensalizados 

In [ ]:
# Renomeando a coluna prevista
df_selic_projecao = projecao_selic.rename(columns={'selic_prevista': 'valor'})
df_ipca_projecao = projecao_ipca.rename(columns={'ipca_previsto': 'valor'})

In [ ]:
# Adicionar coluna "indicador" com valor fixo "SELIC"
df_selic_projecao['indicador'] = 'SELIC'

# Adicionar coluna "indicador" com valor fixo "IPCA"
df_ipca_projecao['indicador'] = 'IPCA'

In [ ]:
# Concatenando os dataframes 
df_macro_projecoes = pd.concat([df_selic_projecao, df_ipca_projecao])

# Garantir que o índice é datetime
df_macro_projecoes.index = pd.to_datetime(df_macro_projecoes.index)

# Ordenar o índice
df_macro_projecoes = df_macro_projecoes.sort_index()

# colocando limite de data menores que 1º de janeiro de 2030.
df_resultado_macro_projecoes = df_macro_projecoes[df_macro_projecoes.index < '2030-01-01']

# Reorganizar as colunas na ordem
df_resultado_macro_projecoes = df_resultado_macro_projecoes[['valor', 'indicador', 'limite_inferior', 'limite_superior']]

## Junção dos DataFrames

In [ ]:
# Concatenando os dataframes 
df_concat_resultado_indicadores = pd.concat([df_resultado_macro_projecoes, df_macro_observados], axis=0, sort=False)

# Garantir que o índice é datetime
df_concat_resultado_indicadores.index = pd.to_datetime(df_concat_resultado_indicadores.index)

# Ordenar o índice
df_concat_resultado_indicadores = df_concat_resultado_indicadores.sort_index()